In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Ready to go")

In [ ]:
import pandas as pd

df = pd.read_csv('/Users/adambrann/Downloads/sleep_hrv_sample.csv')
print(df.shape)
df.head()

In [ ]:
print(df.shape)

In [ ]:
print(df.isnull().sum())

In [ ]:
df.describe()

In [ ]:
df['date'] = pd.to_datetime(df['date'])
print(df.dtypes)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(df['date'], df['total_sleep_hours'], color='steelblue', linewidth=1.5)
plt.axhline(y=8, color='red', linestyle='--', linewidth=1, label='Target (8hrs)')
plt.title('Sleep duration over time')
plt.xlabel('Date')
plt.ylabel('Hours')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['date'], df['hrv_rmssd'], color='teal', linewidth=1.5)
plt.title('HRV (RMSSD) over time')
plt.xlabel('Date')
plt.ylabel('HRV (ms)')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))

ax1.fill_between(df['date'], df['sleep_debt'], alpha=0.2, color='coral')
ax1.plot(df['date'], df['sleep_debt'], color='coral', linewidth=1.5, label='Sleep debt')
ax1.set_ylabel('Sleep debt (hrs)', color='coral')
ax1.tick_params(axis='y', labelcolor='coral')

ax2 = ax1.twinx()
ax2.plot(df['date'], df['hrv_rmssd'], color='teal', linewidth=1.5, label='HRV')
ax2.set_ylabel('HRV (ms)', color='teal')
ax2.tick_params(axis='y', labelcolor='teal')

plt.title('Sleep debt vs HRV over time')
fig.tight_layout()
plt.show()

In [ ]:
from scipy import stats

r, p = stats.pearsonr(df['sleep_debt'], df['hrv_rmssd'])
print(f"Correlation: {r:.2f}")
print(f"P-value: {p:.3f}")
print(f"Significant: {'Yes' if p < 0.05 else 'No'}")

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df['sleep_debt'], df['hrv_rmssd'], 
            alpha=0.6, color='steelblue', edgecolors='white', linewidth=0.5)

# add a trend line
m, b, r, p, se = stats.linregress(df['sleep_debt'], df['hrv_rmssd'])
x_line = np.linspace(df['sleep_debt'].min(), df['sleep_debt'].max(), 100)
plt.plot(x_line, m * x_line + b, color='coral', linewidth=2, label=f'r = {r:.2f}')

plt.title('Sleep debt vs HRV')
plt.xlabel('Sleep debt (hrs)')
plt.ylabel('HRV RMSSD (ms)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def sleep_debt_model(sleep_series, target=8.0, ti=1.2, td=0.85):
    """
    Asymmetric sleep debt model based on Borbély two-process framework.
    Ti > Td: debt accumulates faster than it clears.
    
    Parameters:
    -----------
    sleep_series : list of nightly sleep hours
    target       : sleep target in hours (default 8.0)
    ti           : accumulation rate - how fast debt builds (default 1.2)
    td           : decay rate - how fast debt clears (default 0.85)
    
    Returns:
    --------
    list of daily debt values
    """
    debt = []
    current_debt = 0
    
    for sleep in sleep_series:
        deficit = target - sleep
        if deficit > 0:
            # undersleeping — debt accumulates faster
            current_debt = (current_debt * td) + (deficit * ti)
        else:
            # oversleeping — debt clears at standard decay rate
            current_debt = max(0, (current_debt * td) + deficit)
        debt.append(current_debt)
    
    return debt

# run it on your data
df['sleep_debt_refined'] = sleep_debt_model(df['total_sleep_hours'])
print("Model run successfully")
print(df[['date', 'sleep_debt', 'sleep_debt_refined']].head(10))

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['date'], df['sleep_debt'], color='coral', 
         linewidth=1.5, linestyle='--', label='Original model')
plt.plot(df['date'], df['sleep_debt_refined'], color='crimson', 
         linewidth=1.5, label='Refined model (asymmetric)')
plt.title('Sleep debt: original vs refined model')
plt.xlabel('Date')
plt.ylabel('Debt (hours)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy import stats

r_original, p_original = stats.pearsonr(df['sleep_debt'], df['hrv_rmssd'])
r_refined, p_refined = stats.pearsonr(df['sleep_debt_refined'], df['hrv_rmssd'])

print(f"Original model  — r: {r_original:.3f}, p: {p_original:.4f}")
print(f"Refined model   — r: {r_refined:.3f}, p: {p_refined:.4f}")
print(f"Improvement in r: {abs(r_refined) - abs(r_original):.3f}")

## Model Comparison Note
The refined asymmetric model shows slightly weaker correlation 
on simulated data (r = -0.865 vs -0.891), expected since the 
simulation was generated using simple decay kinetics.

Hypothesis: the asymmetric model will outperform on real 
wearable data where debt accumulation kinetics reflect actual 
human physiology (Ti > Td per Borbély two-process framework).

This comparison will be rerun once personal device data is 
collected — the difference in r values will serve as empirical 
evidence for or against the asymmetric assumption.

In [ ]:
import numpy as np
from itertools import product

best_r = 0
best_ti = 1.2
best_td = 0.85
results = []

ti_range = np.arange(0.9, 1.6, 0.1)
td_range = np.arange(0.70, 0.96, 0.05)

for ti, td in product(ti_range, td_range):
    debt_test = sleep_debt_model(df['total_sleep_hours'], ti=ti, td=td)
    r, p = stats.pearsonr(debt_test, df['hrv_rmssd'])
    results.append({'ti': round(ti,2), 'td': round(td,2), 
                    'r': round(r,3), 'abs_r': round(abs(r),3)})
    if abs(r) > abs(best_r):
        best_r = r
        best_ti = ti
        best_td = td

print(f"Best Ti: {best_ti:.2f}")
print(f"Best Td: {best_td:.2f}")
print(f"Best correlation: {best_r:.3f}")

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
pivot = results_df.pivot(index='ti', columns='td', values='abs_r')

plt.figure(figsize=(10, 6))
im = plt.imshow(pivot, aspect='auto', cmap='RdYlGn')
plt.colorbar(im, label='|r| with HRV')
plt.xticks(range(len(pivot.columns)), 
           [f'{x:.2f}' for x in pivot.columns])
plt.yticks(range(len(pivot.index)), 
           [f'{x:.2f}' for x in pivot.index])
plt.xlabel('Td (decay rate)')
plt.ylabel('Ti (accumulation rate)')
plt.title('Grid search: HRV correlation across Ti/Td combinations')
plt.tight_layout()
plt.show()

In [ ]:
best_r = 0
best_ti = 1.5
best_td = 0.85
results = []

ti_range = np.arange(0.9, 2.5, 0.1)  # extended upper bound
td_range = np.arange(0.70, 0.96, 0.05)

for ti, td in product(ti_range, td_range):
    debt_test = sleep_debt_model(df['total_sleep_hours'], ti=ti, td=td)
    r, p = stats.pearsonr(debt_test, df['hrv_rmssd'])
    results.append({'ti': round(ti,2), 'td': round(td,2), 
                    'r': round(r,3), 'abs_r': round(abs(r),3)})
    if abs(r) > abs(best_r):
        best_r = r
        best_ti = ti
        best_td = td

print(f"Best Ti: {best_ti:.2f}")
print(f"Best Td: {best_td:.2f}")
print(f"Best correlation: {best_r:.3f}")

## Grid Search Note
Ti consistently hits the upper boundary of the search range 
regardless of how far it is extended, indicating the simulated 
data cannot constrain the accumulation parameter — expected 
since the simulation was generated with symmetric kinetics.

Td stabilizes at 0.85 across all searches, suggesting the 
decay parameter is identifiable even on simulated data.

Planned: rerun grid search on real wearable data where 
asymmetric accumulation kinetics should produce a true Ti 
optimum. The point at which Ti stops improving correlation 
will represent a personally calibrated accumulation rate.

In [ ]:
def calibrate_model(sleep_series, hrv_series, 
                    ti_range=np.arange(0.9, 2.5, 0.1),
                    td_range=np.arange(0.70, 0.96, 0.05),
                    min_days=14):
    """
    Automatically calibrates Ti and Td against HRV data.
    Requires minimum 14 days before attempting calibration.
    
    Parameters:
    -----------
    sleep_series : list of nightly sleep hours
    hrv_series   : list of nightly HRV readings
    ti_range     : accumulation rates to search
    td_range     : decay rates to search
    min_days     : minimum days required before calibrating
    
    Returns:
    --------
    dict with best_ti, best_td, best_r, status, days_used
    """
    days = len(sleep_series)
    
    if days < min_days:
        return {
            'best_ti': 1.2,
            'best_td': 0.85,
            'best_r': None,
            'status': f'Insufficient data — using defaults ({days}/{min_days} days)',
            'days_used': days
        }
    
    best_r = 0
    best_ti = 1.2
    best_td = 0.85
    
    for ti, td in product(ti_range, td_range):
        debt_test = sleep_debt_model(sleep_series, ti=ti, td=td)
        r, p = stats.pearsonr(debt_test, hrv_series)
        if abs(r) > abs(best_r):
            best_r = r
            best_ti = ti
            best_td = td
    
    status = 'Calibrated' if days < 30 else 'Well calibrated' if days < 60 else 'Fully calibrated'
    
    return {
        'best_ti': round(best_ti, 2),
        'best_td': round(best_td, 2),
        'best_r': round(best_r, 3),
        'status': status,
        'days_used': days
    }

In [ ]:
params = calibrate_model(
    df['total_sleep_hours'].tolist(),
    df['hrv_rmssd'].tolist()
)

print(f"Status    : {params['status']}")
print(f"Days used : {params['days_used']}")
print(f"Best Ti   : {params['best_ti']}")
print(f"Best Td   : {params['best_td']}")
print(f"Best r    : {params['best_r']}")

In [ ]:
calibration_log = []

for n in range(14, len(df)+1, 7):  # every 7 days from day 14
    result = calibrate_model(
        df['total_sleep_hours'].tolist()[:n],
        df['hrv_rmssd'].tolist()[:n]
    )
    result['day'] = n
    calibration_log.append(result)

cal_df = pd.DataFrame(calibration_log)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(cal_df['day'], cal_df['best_ti'], 
             color='steelblue', linewidth=1.5, marker='o', markersize=4)
axes[0].set_ylabel('Ti (accumulation)')
axes[0].set_title('Parameter stability as data accumulates')
axes[0].axhline(y=params['best_ti'], color='steelblue', 
                linestyle='--', alpha=0.4, label='Final value')
axes[0].legend()

axes[1].plot(cal_df['day'], cal_df['best_td'], 
             color='coral', linewidth=1.5, marker='o', markersize=4)
axes[1].set_ylabel('Td (decay)')
axes[1].axhline(y=params['best_td'], color='coral', 
                linestyle='--', alpha=0.4, label='Final value')
axes[1].legend()

axes[2].plot(cal_df['day'], cal_df['best_r'].abs(), 
             color='teal', linewidth=1.5, marker='o', markersize=4)
axes[2].set_ylabel('|r| with HRV')
axes[2].set_xlabel('Days of data')
axes[2].axhline(y=abs(params['best_r']), color='teal', 
                linestyle='--', alpha=0.4, label='Final value')
axes[2].legend()

plt.tight_layout()
plt.show()

## Calibration Stability Plot
Parameters show no evolution on simulated data — expected, 
since HRV was generated with fixed symmetric kinetics giving 
identical optimal parameters at all data lengths.

On real wearable data, early parameter instability followed 
by convergence will indicate the minimum days required for 
reliable personal calibration — a key practical finding 
for the project.

In [ ]:
def weighted_update(current_ti, current_td,
                    sleep_series, hrv_series,
                    weight=0.3, lookback=21,
                    min_r=0.4):
    """
    Updates Ti and Td by blending recent calibration with
    existing parameters using exponential weighted averaging.
    
    Parameters:
    -----------
    current_ti   : existing Ti parameter
    current_td   : existing Td parameter
    sleep_series : full sleep history as list
    hrv_series   : full HRV history as list
    weight       : influence of new data (0-1), default 0.3
    lookback     : days of recent data to calibrate against
    min_r        : minimum |r| required to accept update
    
    Returns:
    --------
    dict with updated parameters and diagnostics
    """
    recent_sleep = sleep_series[-lookback:]
    recent_hrv = hrv_series[-lookback:]
    
    new_params = calibrate_model(recent_sleep, recent_hrv,
                                 min_days=lookback)
    
    # guard 1 — skip if correlation is positive
    if new_params['best_r'] > 0:
        return {
            'updated_ti': current_ti,
            'updated_td': current_td,
            'new_ti': new_params['best_ti'],
            'new_td': new_params['best_td'],
            'recent_r': new_params['best_r'],
            'weight_applied': 0,
            'days_in_update': lookback,
            'update_accepted': False,
            'skip_reason': 'Positive correlation — update rejected'
        }
    
    # guard 2 — skip if correlation is too weak
    if abs(new_params['best_r']) < min_r:
        return {
            'updated_ti': current_ti,
            'updated_td': current_td,
            'new_ti': new_params['best_ti'],
            'new_td': new_params['best_td'],
            'recent_r': new_params['best_r'],
            'weight_applied': 0,
            'days_in_update': lookback,
            'update_accepted': False,
            'skip_reason': f'Weak correlation ({abs(new_params["best_r"]):.2f} < {min_r}) — update rejected'
        }
    
    # both guards passed — apply weighted update
    updated_ti = (current_ti * (1 - weight)) + (new_params['best_ti'] * weight)
    updated_td = (current_td * (1 - weight)) + (new_params['best_td'] * weight)
    
    # parameter bounds — prevent drift beyond physiological range
    updated_ti = float(np.clip(updated_ti, 1.0, 2.0))
    updated_td = float(np.clip(updated_td, 0.75, 0.95))
    
    return {
        'updated_ti': round(updated_ti, 3),
        'updated_td': round(updated_td, 3),
        'new_ti': new_params['best_ti'],
        'new_td': new_params['best_td'],
        'recent_r': new_params['best_r'],
        'weight_applied': weight,
        'days_in_update': lookback,
        'update_accepted': True,
        'skip_reason': None
    }    

In [ ]:
# start with literature defaults
ti = 1.2
td = 0.85
update_log = []

sleep_list = df['total_sleep_hours'].tolist()
hrv_list = df['hrv_rmssd'].tolist()

for day in range(28, len(df)+1, 7):
    result = weighted_update(
        current_ti=ti,
        current_td=td,
        sleep_series=sleep_list[:day],
        hrv_series=hrv_list[:day]
    )
    ti = result['updated_ti']
    td = result['updated_td']
    result['day'] = day
    update_log.append(result)
    
    accepted = '✓ accepted' if result['update_accepted'] else f'✗ skipped — {result["skip_reason"]}'
    print(f"Day {day:3d} | Ti: {ti:.3f} | Td: {td:.3f} | r: {result['recent_r']:.3f} | {accepted}")

update_df = pd.DataFrame(update_log)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(update_df['day'], update_df['updated_ti'], 
             color='steelblue', linewidth=1.5, marker='o', markersize=4)
axes[0].axhline(y=1.2, color='steelblue', linestyle='--', 
                alpha=0.4, label='Literature default')
axes[0].set_ylabel('Ti (accumulation)')
axes[0].set_title('Weighted parameter evolution over time')
axes[0].legend()

axes[1].plot(update_df['day'], update_df['updated_td'], 
             color='coral', linewidth=1.5, marker='o', markersize=4)
axes[1].axhline(y=0.85, color='coral', linestyle='--', 
                alpha=0.4, label='Literature default')
axes[1].set_ylabel('Td (decay)')
axes[1].set_xlabel('Days of data')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
final_ti = update_df['updated_ti'].iloc[-1]
final_td = update_df['updated_td'].iloc[-1]

df['sleep_debt_calibrated'] = sleep_debt_model(
    df['total_sleep_hours'], 
    ti=final_ti, 
    td=final_td
)

r_original, _ = stats.pearsonr(df['sleep_debt'], df['hrv_rmssd'])
r_refined, _ = stats.pearsonr(df['sleep_debt_refined'], df['hrv_rmssd'])
r_calibrated, _ = stats.pearsonr(df['sleep_debt_calibrated'], df['hrv_rmssd'])

print(f"Original model    r: {r_original:.3f}")
print(f"Refined model     r: {r_refined:.3f}")
print(f"Calibrated model  r: {r_calibrated:.3f}")
print(f"\nFinal Ti: {final_ti:.3f}")
print(f"Final Td: {final_td:.3f}")

## Weighted EWMA Calibration — Current Status

### Implementation
Self-calibrating parameter update system using Exponential 
Weighted Moving Average (EWMA) blending. Weekly calibration 
runs against a 21-day lookback window, blending new estimates 
with existing parameters at a configurable weight.

### Safeguards Implemented
- **Positive correlation guard** — rejects updates where recent 
  r > 0, preventing anomalous recovery windows from corrupting 
  parameters
- **Weak signal guard** — rejects updates where |r| < 0.4, 
  ensuring only meaningful calibration windows influence parameters
- **Parameter bounds** — hard physiological limits prevent drift:
  - Ti constrained to [1.0, 2.0]
  - Td constrained to [0.75, 0.95]

### Simulated Data Limitations
Weekly updates show consistent parameter drift on simulated data 
(Ti: 1.560 → 1.058, Td: 0.820 → 0.750 over 5 update cycles), 
expected since simulated HRV was generated with symmetric kinetics 
producing uniform directional signal across all 21-day windows.

Key observations:
- Td stabilizes at lower bound (0.750) — bound functioning correctly
- Ti continues drifting — insufficient signal variation in 
  simulated data to produce natural equilibrium
- All 5 updates accepted — correlation consistently negative 
  and above minimum threshold

### Planned Refinements on Real Data
1. Reduce update weight from 0.3 to 0.15 for more conservative 
   parameter evolution
2. Extend update frequency from weekly to biweekly to reduce 
   noise sensitivity
3. Rerun full calibration sequence — expect natural Ti/Td 
   equilibrium when genuine physiological signal replaces 
   simulated data
4. Validate whether asymmetric model (Ti > Td) outperforms 
   simple decay on personal HRV data — core hypothesis of 
   the project

### Model Versions Comparison (Simulated Data)
| Model | r with HRV | Notes |
|-------|-----------|-------|
| Original (simple decay) | -0.891 | Symmetric, fixed td=0.85 |
| Refined (asymmetric) | -0.865 | Fixed Ti=1.2, Td=0.85 |
| Calibrated (EWMA) | -0.732 | Drifting on simulated data |

Note: calibrated model expected to outperform on real data 
where asymmetric kinetics reflect genuine physiology.